# Action Anticipation

---

**How this works step by step:**

| Step | What happens |
|---|---|
| Frame $t$ | Only frames $0...t$ are visible |
| Build $\mathbf{P}_t$ | Causal (Window) graph from past frames only |
| Apply $\mathbf{P}_t^{\nabla}$ | Propagate from last obs frame forward |
| Predict frame $t+\nabla$ | Using only past information |
| Evaluate | mAP on ALL, FREQ, RARE |


In [1]:
# ============================================================
# OPTION C — CAUSAL ANTICIPATION
# For each frame t, build P using only
# past frames 0...t then anticipate t+delta
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import scipy.sparse as sp
from sklearn.metrics import average_precision_score
import os

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# PATHS
# ============================================================
FEATURES_CSV_EPIC  = "../Features/EPIC/P01_04_vgg_fused_features.csv"
LABELS_CSV_EPIC    = "../Labels/EPIC/P01_04_vgg_fused_labeled.csv"
AFTER_PROP_EPIC    = "../Prediction/EPIC/P01_04_vgg_fused_after_propagation.csv"
BEFORE_PROP_EPIC   = "../Prediction/EPIC/P01_04_vgg_fused_before_propagation.csv"
SAVE_EPIC          = "../Metrics/EPIC/"
os.makedirs(SAVE_EPIC, exist_ok=True)

FEATURES_CSV_ADL   = "../Features/ADL/P_11_rgb_features.csv"
LABELS_CSV_ADL     = "../Labels/ADL/P_11_labeled.csv"
AFTER_PROP_ADL     = "../Prediction/ADL/P_11_after_propagation.csv"
BEFORE_PROP_ADL    = "../Prediction/ADL/P_11_before_propagation.csv"
SAVE_ADL           = "../Metrics/ADL/"
os.makedirs(SAVE_ADL, exist_ok=True)

# ============================================================
# METRICS
# ============================================================
def compute_split_mAP(y_true, y_pred,
                      class_indices):
    AP = []
    for c in class_indices:
        if y_true[:, c].sum() == 0:
            continue
        ap = average_precision_score(
            y_true[:, c], y_pred[:, c])
        AP.append(ap)
    return np.mean(AP) if len(AP) > 0 \
        else 0.0

def get_freq_rare_splits(y_true,
                         threshold=0.5):
    class_counts = y_true.sum(axis=0)
    num_classes  = y_true.shape[1]
    sorted_idx   = np.argsort(
        class_counts)[::-1]
    split_point  = int(
        num_classes * threshold)
    freq_classes = sorted_idx[
        :split_point].tolist()
    rare_classes = sorted_idx[
        split_point:].tolist()
    freq_classes = [
        c for c in freq_classes
        if class_counts[c] > 0]
    rare_classes = [
        c for c in rare_classes
        if class_counts[c] > 0]
    return freq_classes, rare_classes

# ============================================================
# CAUSAL TRANSITION MATRIX BUILDER
# Build P using only past frames 0...t
# ============================================================
def build_causal_P(features, t, k=5):
    """
    Build transition matrix using only
    frames 0 to t (inclusive).
    Returns P as numpy array (t+1 x t+1)
    """
    feat_t = torch.from_numpy(
        features[:t+1]
    ).float().to(DEVICE)
    feat_t = F.normalize(feat_t, dim=1)

    # Cosine similarity (t+1 x t+1)
    S      = torch.matmul(
        feat_t, feat_t.T
    ).cpu().numpy()

    # k-NN sparsification
    A      = np.zeros_like(S)
    k_act  = min(k, t+1)
    for i in range(t+1):
        idx       = np.argpartition(
            S[i], -k_act)[-k_act:]
        A[i, idx] = S[i, idx]

    # Mutual k-NN
    A      = A * A.T
    np.fill_diagonal(A, 0.0)

    # Row normalize
    row_sum = A.sum(axis=1, keepdims=True)
    P       = A / np.maximum(
        row_sum, 1e-8)
    return P

# ============================================================
# CAUSAL ANTICIPATION — CARRW
# ============================================================
def causal_carrw(logits_t, P_t,
                 alpha=0.5, steps=5):
    """
    Apply CARRW on observation window
    logits_t: numpy (t+1 x C)
    P_t     : numpy (t+1 x t+1)
    Returns refined predictions (t+1 x C)
    """
    Y           = torch.sigmoid(
        torch.from_numpy(logits_t
        ).float().to(DEVICE))
    P_tensor    = torch.from_numpy(
        P_t).float().to(DEVICE)
    confidence  = torch.abs(Y - 0.5) * 2
    uncertainty = 1 - confidence

    for _ in range(steps):
        Y = Y + alpha * uncertainty * (
            torch.matmul(P_tensor, Y) - Y)
    return torch.clamp(
        Y, 0, 1).cpu().numpy()

# ============================================================
# FIXED — Load CSVs once, process all frames
# ============================================================

def evaluate_causal_anticipation(
        features_csv,
        labels_csv,
        after_prop_csv,
        before_prop_csv,
        dataset_name,
        delta_steps=[15, 30, 45, 60, 90, 120],
        k=5,
        min_obs=100,
        stride=10):

    print(f"\n{'='*60}")
    print(f"Dataset: {dataset_name}")
    print(f"{'='*60}")

    # ---- Load ALL data once ----
    feat_df      = pd.read_csv(features_csv)
    features     = feat_df.drop(
                       columns=["frame_id"]
                   ).values.astype(np.float32)

    labels_df    = pd.read_csv(labels_csv)
    y_true       = labels_df.drop(
                       columns=["frame_id"]
                   ).values.astype(np.float32)

    after_df     = pd.read_csv(after_prop_csv)
    Y_star_full  = after_df.drop(
                       columns=["frame_id"]
                   ).values.astype(np.float32)

    before_df    = pd.read_csv(before_prop_csv)
    Y_rw_full    = before_df.drop(
                       columns=["frame_id"]
                   ).values.astype(np.float32)

    T            = y_true.shape[0]
    NUM_CLASSES  = y_true.shape[1]
    all_classes  = list(range(NUM_CLASSES))

    freq_classes, rare_classes = \
        get_freq_rare_splits(y_true)

    print(f"Total frames  : {T}")
    print(f"Total classes : {NUM_CLASSES}")
    print(f"FREQ classes  : {len(freq_classes)}")
    print(f"RARE classes  : {len(rare_classes)}")
    print(f"Stride        : {stride}")
    print(f"Min obs       : {min_obs}")

    all_results = []

    for delta in delta_steps:

        print(f"\n  Processing ∇={delta}...")

        y_pred_c_list = []
        y_pred_r_list = []
        y_true_list   = []
        frames_done   = 0

        for t in range(
                min_obs,
                T - delta,
                stride):

            t_future = t + delta
            if t_future >= T:
                break

            # ---- Build causal P ----
            # Only frames 0...t
            feat_obs  = features[:t+1]
            feat_t    = torch.from_numpy(
                feat_obs).float().to(DEVICE)
            feat_t    = F.normalize(
                feat_t, dim=1)

            S         = torch.matmul(
                feat_t, feat_t.T
            ).cpu().numpy()

            A         = np.zeros_like(S)
            k_act     = min(k, t+1)
            for i in range(t+1):
                idx       = np.argpartition(
                    S[i], -k_act)[-k_act:]
                A[i, idx] = S[i, idx]

            A         = A * A.T
            np.fill_diagonal(A, 0.0)
            row_sum   = A.sum(
                axis=1, keepdims=True)
            P_t       = A / np.maximum(
                row_sum, 1e-8)

            P_tensor  = torch.from_numpy(
                P_t.astype(np.float32)
            ).to(DEVICE)

            # ---- P^delta ----
            P_delta   = torch.linalg\
                .matrix_power(
                    P_tensor, delta)

            # ---- Last row of P^delta ----
            # Shape: (1 x t+1)
            p_last    = P_delta[-1:, :]

            # ---- Obs window predictions ----
            # Use preloaded data — no CSV read
            Y_star_obs = torch.from_numpy(
                Y_star_full[:t+1]
            ).float().to(DEVICE)

            Y_rw_obs   = torch.from_numpy(
                Y_rw_full[:t+1]
            ).float().to(DEVICE)

            # ---- Anticipate ----
            # (1 x t+1) x (t+1 x C) = (1 x C)
            y_ant_c   = torch.matmul(
                p_last, Y_star_obs
            ).clamp(0, 1).cpu().numpy()[0]

            y_ant_r   = torch.matmul(
                p_last, Y_rw_obs
            ).clamp(0, 1).cpu().numpy()[0]

            y_pred_c_list.append(y_ant_c)
            y_pred_r_list.append(y_ant_r)
            y_true_list.append(
                y_true[t_future])

            frames_done += 1

            # Progress
            if frames_done % 50 == 0:
                print(f"    Processed "
                      f"{frames_done} frames...")

        print(f"  Total frames: {frames_done}")

        if len(y_true_list) == 0:
            continue

        y_pred_c  = np.array(y_pred_c_list)
        y_pred_r  = np.array(y_pred_r_list)
        y_true_ev = np.array(y_true_list)

        # Verify difference
        diff = np.abs(
            y_pred_c - y_pred_r).mean()
        print(f"  Mean diff: {diff:.6f}")

        # Compute mAP splits
        mAP_all_c  = compute_split_mAP(
            y_true_ev, y_pred_c,
            all_classes)
        mAP_freq_c = compute_split_mAP(
            y_true_ev, y_pred_c,
            freq_classes)
        mAP_rare_c = compute_split_mAP(
            y_true_ev, y_pred_c,
            rare_classes
        ) if len(rare_classes) > 0 \
          else 0.0

        mAP_all_r  = compute_split_mAP(
            y_true_ev, y_pred_r,
            all_classes)
        mAP_freq_r = compute_split_mAP(
            y_true_ev, y_pred_r,
            freq_classes)
        mAP_rare_r = compute_split_mAP(
            y_true_ev, y_pred_r,
            rare_classes
        ) if len(rare_classes) > 0 \
          else 0.0

        print(
            f"  RW    → "
            f"ALL={mAP_all_r:.4f}  "
            f"FREQ={mAP_freq_r:.4f}  "
            f"RARE={mAP_rare_r:.4f}"
        )
        print(
            f"  CARRW → "
            f"ALL={mAP_all_c:.4f}  "
            f"FREQ={mAP_freq_c:.4f}  "
            f"RARE={mAP_rare_c:.4f}"
        )
        print(
            f"  Gain  → "
            f"ALL={mAP_all_c-mAP_all_r:+.4f}  "
            f"FREQ={mAP_freq_c-mAP_freq_r:+.4f}  "
            f"RARE={mAP_rare_c-mAP_rare_r:+.4f}"
        )

        all_results.append({
            "Dataset":    dataset_name,
            "Delta":      delta,
            "Num_Frames": frames_done,
            "RW_ALL":     round(mAP_all_r,  4),
            "RW_FREQ":    round(mAP_freq_r, 4),
            "RW_RARE":    round(mAP_rare_r, 4),
            "CARRW_ALL":  round(mAP_all_c,  4),
            "CARRW_FREQ": round(mAP_freq_c, 4),
            "CARRW_RARE": round(mAP_rare_c, 4),
            "Gain_ALL":   round(
                mAP_all_c - mAP_all_r, 4),
            "Gain_FREQ":  round(
                mAP_freq_c - mAP_freq_r, 4),
            "Gain_RARE":  round(
                mAP_rare_c - mAP_rare_r, 4),
        })

    return pd.DataFrame(all_results)

# RUN — EPIC-Kitchens

In [3]:
# ============================================================
# RUN — EPIC-Kitchens
# stride=10 for more frames
# min_obs=50 for faster start
# ============================================================
print("Running EPIC-Kitchens...")
epic_results = evaluate_causal_anticipation(
    features_csv   = FEATURES_CSV_EPIC,
    labels_csv     = LABELS_CSV_EPIC,
    after_prop_csv = AFTER_PROP_EPIC,
    before_prop_csv= BEFORE_PROP_EPIC,
    dataset_name   = "EPIC-Kitchens",
    delta_steps    = [15, 30, 45, 60, 90, 120],
    k              = 5,
    min_obs        = 100,
    stride         = 10
)
epic_results.to_csv(
    f"{SAVE_EPIC}"
    f"P01_04_causal_anticipation.csv",
    index=False
)
print("\nEPIC saved.")

Running EPIC-Kitchens...

Dataset: EPIC-Kitchens
Total frames  : 6180
Total classes : 29
FREQ classes  : 14
RARE classes  : 15
Stride        : 10
Min obs       : 100

  Processing ∇=15...
    Processed 50 frames...
    Processed 100 frames...
    Processed 150 frames...
    Processed 200 frames...
    Processed 250 frames...
    Processed 300 frames...
    Processed 350 frames...
    Processed 400 frames...
    Processed 450 frames...
    Processed 500 frames...
    Processed 550 frames...
    Processed 600 frames...
  Total frames: 607
  Mean diff: 0.002979
  RW    → ALL=0.8397  FREQ=0.9270  RARE=0.7582
  CARRW → ALL=0.8404  FREQ=0.9284  RARE=0.7583
  Gain  → ALL=+0.0007  FREQ=+0.0014  RARE=+0.0001

  Processing ∇=30...
    Processed 50 frames...
    Processed 100 frames...
    Processed 150 frames...
    Processed 200 frames...
    Processed 250 frames...
    Processed 300 frames...
    Processed 350 frames...
    Processed 400 frames...
    Processed 450 frames...
    Processed 500 

# RUN — ADL

In [4]:
# ============================================================
# RUN — ADL
# ============================================================
print("\nRunning ADL...")
adl_results = evaluate_causal_anticipation(
    features_csv   = FEATURES_CSV_ADL,
    labels_csv     = LABELS_CSV_ADL,
    after_prop_csv = AFTER_PROP_ADL,
    before_prop_csv= BEFORE_PROP_ADL,
    dataset_name   = "ADL",
    delta_steps    = [15, 30, 45, 60],
    k              = 5,
    min_obs        = 100,
    stride         = 10
)
adl_results.to_csv(
    f"{SAVE_ADL}"
    f"P_11_causal_anticipation.csv",
    index=False
)
print("\nADL saved.")
print("\nAll done!")


Running ADL...

Dataset: ADL
Total frames  : 14775
Total classes : 32
FREQ classes  : 8
RARE classes  : 0
Stride        : 10
Min obs       : 100

  Processing ∇=15...
    Processed 50 frames...
    Processed 100 frames...
    Processed 150 frames...
    Processed 200 frames...
    Processed 250 frames...
    Processed 300 frames...
    Processed 350 frames...
    Processed 400 frames...
    Processed 450 frames...
    Processed 500 frames...
    Processed 550 frames...
    Processed 600 frames...
    Processed 650 frames...


KeyboardInterrupt: 

# COMBINED ANTICIPATION QUALITATIVE FIGURE

In [3]:
# ============================================================
# COMBINED ANTICIPATION QUALITATIVE FIGURE
# Top: Timeline with ground truth action bars
# Bottom: Anticipated confidence curves at different horizons
# With frame snapshots showing GT vs Anticipated labels
# ============================================================

import numpy as np
import pandas as pd
import torch
import scipy.sparse as sp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# PATHS
# ============================================================
LABELS_CSV    = "../Labels/EPIC/P01_04_vgg_fused_labeled.csv"
AFTER_PROP    = "../Prediction/EPIC/P01_04_vgg_fused_after_propagation.csv"
BEFORE_PROP   = "../Prediction/EPIC/P01_04_vgg_fused_before_propagation.csv"
FEATURES_CSV  = "../Features/EPIC/P01_04_vgg_fused_features.csv"
ANNOTATION    = r"D:\Datasets\Datasets\EPIC_Kitchen\Label\P01_04.csv"
GRAPH_PATH    = "../Graph/EPIC/P01_04_vgg_fused_transition_matrix.npz"
SAVE_DIR      = "../Qualitative/EPIC/"
os.makedirs(SAVE_DIR, exist_ok=True)

FPS           = 60
# Time horizons in seconds → frame deltas
TIME_HORIZONS = [0.25, 0.50, 0.75, 1.00]
DELTA_STEPS   = [int(t * FPS) for t in TIME_HORIZONS]
# [15, 30, 45, 60]

# ============================================================
# LOAD
# ============================================================
labels_df    = pd.read_csv(LABELS_CSV)
y_true       = labels_df.drop(
                   columns=["frame_id"]
               ).values.astype(np.float32)
frame_ids    = labels_df["frame_id"].values
NUM_FRAMES, NUM_CLASSES = y_true.shape

after_df     = pd.read_csv(AFTER_PROP)
Y_star       = after_df.drop(
                   columns=["frame_id"]
               ).values.astype(np.float32)

before_df    = pd.read_csv(BEFORE_PROP)
Y_rw         = before_df.drop(
                   columns=["frame_id"]
               ).values.astype(np.float32)

feat_df      = pd.read_csv(FEATURES_CSV)
features     = feat_df.drop(
                   columns=["frame_id"]
               ).values.astype(np.float32)

ann_df       = pd.read_csv(ANNOTATION)
ann_map      = ann_df[[
    "ActionLabel","ActionName"]
].drop_duplicates()
action_map   = dict(zip(
    ann_map["ActionLabel"].astype(int),
    ann_map["ActionName"]
))

P_sparse     = sp.load_npz(GRAPH_PATH)
P            = P_sparse.toarray(
               ).astype(np.float32)
P_tensor     = torch.from_numpy(P).to(DEVICE)

# ============================================================
# SMOOTH
# ============================================================
def smooth(arr, window=5):
    kernel = np.ones(window) / window
    return np.convolve(arr, kernel, mode='same')

# ============================================================
# BUILD CAUSAL P AT FRAME t
# ============================================================
def build_causal_P(features, t, k=5):
    import torch.nn.functional as F
    feat_t = torch.from_numpy(
        features[:t+1]).float().to(DEVICE)
    feat_t = F.normalize(feat_t, dim=1)
    S      = torch.matmul(
        feat_t, feat_t.T).cpu().numpy()
    A      = np.zeros_like(S)
    k_act  = min(k, t+1)
    for i in range(t+1):
        idx       = np.argpartition(
            S[i], -k_act)[-k_act:]
        A[i, idx] = S[i, idx]
    A      = A * A.T
    np.fill_diagonal(A, 0.0)
    row_sum = A.sum(axis=1, keepdims=True)
    P_t    = A / np.maximum(row_sum, 1e-8)
    return torch.from_numpy(
        P_t.astype(np.float32)).to(DEVICE)

# ============================================================
# SELECT BEST SEGMENT
# Find a segment with clear action onset
# and multiple co-occurring actions
# ============================================================
ann_full     = pd.read_csv(ANNOTATION)
ann_full["Duration"] = ann_full["EndFrame"] - \
                       ann_full["StartFrame"]

# Find frame with most co-occurring actions
# in a window of 300 frames
best_t       = None
best_count   = 0
WINDOW       = 300

for t in range(200, NUM_FRAMES - 200, 50):
    count = y_true[
        t:t+WINDOW].sum(axis=0
    ).astype(bool).sum()
    if count > best_count:
        best_count = count
        best_t     = t

print(f"Best segment start: {best_t}")
print(f"Active classes: {best_count}")

# Find active classes in this segment
active_mask  = y_true[
    best_t:best_t+WINDOW
].sum(axis=0) > 0
active_cls   = np.where(active_mask)[0][:6]
print(f"Active class indices: {active_cls}")
print(f"Active class names: "
      f"{[action_map.get(c, str(c)) for c in active_cls]}")

# ============================================================
# COMPUTE ANTICIPATED PREDICTIONS
# at observation frame t_obs
# for all time horizons
# ============================================================
t_obs        = best_t + 100  # observation point
BEFORE       = 60
AFTER        = max(DELTA_STEPS) + 60

f_start      = max(0, t_obs - BEFORE)
f_end        = min(NUM_FRAMES,
                   t_obs + AFTER)

idx_s        = int(np.argmin(
    np.abs(frame_ids - f_start)))
idx_e        = int(np.argmin(
    np.abs(frame_ids - f_end)))
frames       = frame_ids[idx_s:idx_e]

print(f"\nObservation point: {t_obs}")
print(f"Window: {f_start} — {f_end}")

# Build causal P at t_obs
print("Building causal P...")
P_obs        = build_causal_P(
    features, t_obs, k=5)
print("Done.")

# Compute anticipated predictions
Y_ant        = {}
for delta, time_h in zip(
        DELTA_STEPS, TIME_HORIZONS):
    P_delta      = torch.linalg\
        .matrix_power(P_obs, delta)
    p_last       = P_delta[-1:, :]
    Y_obs_c      = torch.from_numpy(
        Y_star[:t_obs+1]
    ).float().to(DEVICE)
    y_ant        = torch.matmul(
        p_last, Y_obs_c
    ).clamp(0, 1).cpu().numpy()[0]
    Y_ant[time_h] = y_ant
    print(f"  ∇={delta} ({time_h}s) done")

# ============================================================
# FIGURE — Combined
# Row 1: Ground truth action timeline
# Row 2-5: Anticipated confidence at each horizon
# ============================================================
 
n_rows       = 1 + len(TIME_HORIZONS)
row_heights  = [0.35] + \
               [0.65/len(TIME_HORIZONS)] * \
               len(TIME_HORIZONS)

subplot_titles = ["Ground truth action timeline"] + \
    [f"Anticipated at {t}s ahead"
     for t in TIME_HORIZONS]

fig = make_subplots(
    rows=n_rows, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    subplot_titles=subplot_titles,
    row_heights=row_heights
)

colors_cls   = [
    "#1565C0", "#E53935",
    "#43A047", "#8E24AA",
    "#FB8C00", "#00838F"
]

# ---- ROW 1: Ground truth timeline ----
for ci, cls in enumerate(active_cls):
    gt_vals  = y_true[idx_s:idx_e, cls]
    cname    = action_map.get(
        int(cls), f"class {cls}")
    color    = colors_cls[
        ci % len(colors_cls)]

    # Ground truth bar
    fig.add_trace(
        go.Scatter(
            x=frames,
            y=gt_vals * (ci + 1),
            mode="lines",
            line=dict(
                color=color, width=3),
            name=cname,
            legendgroup=f"cls{cls}",
            showlegend=True,
            hovertemplate=f"{cname}<br>"
                          f"frame: %{{x}}<br>"
                          f"GT: %{{y:.2f}}"
        ),
        row=1, col=1
    )

# Observation point line
fig.add_vline(
    x=frame_ids[
        np.argmin(np.abs(
            frame_ids - t_obs))],
    line=dict(
        color="black",
        width=2, dash="dash"),
    row=1, col=1
)

# ---- ROWS 2-5: Anticipated confidence ----
for ri, (time_h, delta) in enumerate(
        zip(TIME_HORIZONS, DELTA_STEPS),
        start=2):

    y_ant    = Y_ant[time_h]
    t_future = t_obs + delta
    t_future_frame = frame_ids[
        min(np.argmin(np.abs(
            frame_ids - t_future)),
            len(frame_ids)-1)]

    for ci, cls in enumerate(active_cls):
        cname = action_map.get(
            int(cls), f"class {cls}")
        color = colors_cls[
            ci % len(colors_cls)]

        # Ground truth in this window
        gt_smooth = smooth(
            y_true[idx_s:idx_e, cls],
            window=5)
        fig.add_trace(
            go.Scatter(
                x=frames,
                y=gt_smooth,
                mode="lines",
                line=dict(
                    color=color,
                    width=1.0,
                    dash="dot"),
                name=f"GT {cname}",
                legendgroup=f"cls{cls}",
                showlegend=False,
                opacity=0.4,
                hoverinfo="skip"
            ),
            row=ri, col=1
        )

        # Anticipated confidence
        # — horizontal line at future point
        fig.add_trace(
            go.Scatter(
                x=[t_future_frame,
                   t_future_frame],
                y=[0, y_ant[cls]],
                mode="lines",
                line=dict(
                    color=color,
                    width=3),
                name=f"Ant {cname}",
                legendgroup=f"cls{cls}",
                showlegend=False,
                hovertemplate=
                    f"{cname}<br>"
                    f"Anticipated: "
                    f"{y_ant[cls]:.3f}<br>"
                    f"GT: "
                    f"{y_true[min(np.argmin(np.abs(frame_ids - t_future)), len(frame_ids)-1), cls]:.3f}"
            ),
            row=ri, col=1
        )

        # GT at future point marker
        gt_future = y_true[
            min(np.argmin(np.abs(
                frame_ids - t_future)),
                len(frame_ids)-1), cls]
        fig.add_trace(
            go.Scatter(
                x=[t_future_frame],
                y=[gt_future],
                mode="markers",
                marker=dict(
                    color=color,
                    size=10,
                    symbol="circle",
                    line=dict(
                        color="white",
                        width=1.5)),
                name=f"GT@future {cname}",
                legendgroup=f"cls{cls}",
                showlegend=False,
                hovertemplate=
                    f"GT at {time_h}s: "
                    f"{gt_future:.3f}"
            ),
            row=ri, col=1
        )

    # Observation point line
    fig.add_vline(
        x=frame_ids[
            np.argmin(np.abs(
                frame_ids - t_obs))],
        line=dict(
            color="black",
            width=1.5,
            dash="dash"),
        row=ri, col=1
    )

    # Future point line
    fig.add_vline(
        x=t_future_frame,
        line=dict(
            color="gray",
            width=1.5,
            dash="dot"),
        row=ri, col=1
    )

    # Shaded anticipation region
    fig.add_vrect(
        x0=frame_ids[
            np.argmin(np.abs(
                frame_ids - t_obs))],
        x1=t_future_frame,
        fillcolor="royalblue",
        opacity=0.05,
        layer="below",
        line_width=0,
        row=ri, col=1
    )

    # Y axis label
    fig.update_yaxes(
        title_text="Confidence",
        range=[-0.05, 1.10],
        title_font=dict(size=10),
        row=ri, col=1
    )

# ---- ANNOTATIONS ----
# Observation label
fig.add_annotation(
    x=frame_ids[
        np.argmin(np.abs(
            frame_ids - t_obs))],
    y=1.05,
    xref="x", yref="y",
    text="<b>Obs. point</b>",
    showarrow=False,
    font=dict(size=10, color="black")
)

# ---- LAYOUT ----
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(
        family="Times New Roman",
        size=11
    ),
    height=800,
    width=1000,
    title=dict(
        text="Anticipated vs Ground Truth "
             "Actions at Different Time "
             "Horizons — EPIC-Kitchens (P01_04)",
        font=dict(size=13),
        x=0.5
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.08,
        xanchor="center",
        x=0.5,
        font=dict(size=10),
        bgcolor="rgba(0,0,0,0)",
        borderwidth=0
    ),
    margin=dict(t=60, b=80, l=80, r=40)
)

# Subplot title font
for ann in fig.layout.annotations:
    ann.font = dict(
        size=11,
        family="Times New Roman",
        color="black"
    )

# X axis label on last row
fig.update_xaxes(
    title_text="Frame Index",
    title_font=dict(size=11),
    row=n_rows, col=1
)

# Row 1 y axis
fig.update_yaxes(
    title_text="GT label",
    row=1, col=1
)

# ============================================================
# SAVE
# ============================================================
# fig.write_image(
#     f"{SAVE_DIR}"
#     f"P01_04_anticipated_vs_gt.png",
#     scale=3
# )
# fig.write_image(
#     f"{SAVE_DIR}"
#     f"P01_04_anticipated_vs_gt.eps",
#     scale=3
# )
fig.show()
print("Saved.")

Best segment start: 4000
Active classes: 4
Active class indices: [ 0 17 18 19]
Active class names: ['pull-down cup', 'rinse sink', 'rinse hand', 'close tap']

Observation point: 4100
Window: 4040 — 4220
Building causal P...
Done.
  ∇=15 (0.25s) done
  ∇=30 (0.5s) done
  ∇=45 (0.75s) done
  ∇=60 (1.0s) done


Saved.
